# QRx = b -> Rx =QT*b -> Rx = b_tranformed

# from here its standard backward substitution hence the code is also similar to ref(A)*x = b


In [ ]:
function solutions(Q:: Matrix{<:Number},U:: Matrix{<:Number},b_raw:: Vector{<:Number},r:: Int64)
    b = transpose(Q)*b_raw
    sz = size(U)
    rank = 0
    for i in 1:1:sz[1]
        row_vec = @view U[i,1:sz[2]]
        for j in 1:1:sz[2]
            if row_vec[j] ==0 
                nothing
            else
                rank +=1
                break
            end
        end
    end
    
    particular_vector = zeros(Float64,sz[2],1)
    row_status = zeros(Int64,sz[1],2) 
    # rows of row_status = number of rows
    # first column tells wheather a row is pivot row(1) or free row(0) 
    # second column tells in which column that pivot is 
    column_status = zeros(Int64,sz[2],1)
    # columns of column_status = number of columns in A which is A|b = U
    # first row tells wheather a column is pivot column(1) or free column(0)
    
 @inbounds  for i in 1:1:sz[1]
    @inbounds     for j in 1:1:sz[2]
            if U[i,j] !=0
                row_status[i,1] = 1
                row_status[i,2] = j
                column_status[j] =1
                break
            
            end
        end
    end # this code block is used to find the pivot columns and pivot rows, and store them in row_status and column_status respectively

    @inbounds for i in sz[1]:-1:1
        if row_status[i,1] ==0
             nothing
        else
            pivot_column = row_status[i,2]
            b_comp = b[i]
          @inbounds @simd  for doraemon in pivot_column+1:1:sz[2]
                if column_status[doraemon] == 0
                    nothing
                else
                    b_comp -= U[i,doraemon]*particular_vector[doraemon]
                end
            end
            particular_vector[pivot_column]= b_comp/U[i,pivot_column]
        end
    end  # this code block is used to find the particular solution of the system of linear equations, and store it in particular_vector

     if rank == sz[2]
        return particular_vector
    else 

       linear_combination_of_vectors = Matrix{Float64}[]   
@inbounds for j in 1:1:sz[2]
            if column_status[j] == 0 #= We only care about free columns where column_status[j] == 0
                                         Because the pivot variabels of basis vector are (-1)*M[:,j] where j is a free column
                                        =#
            
            basis_vector = zeros(Float64,sz[2],1)
            basis_vector[j] = 1 # the variable of free column(free variable is set to 1 => this is linear algebra theory)
          @inbounds for i in sz[1]:-1:1
                if row_status[i,1] ==0
                    nothing
                else
                    pivot_column = row_status[i,2]
                    b_comp = 0
                    @inbounds @simd  for doraemon in pivot_column+1:1:sz[2]
                        b_comp -= U[i,doraemon]*basis_vector[doraemon]
                                     end
                    basis_vector[pivot_column]= b_comp/U[i,pivot_column]
                end
            end
            push!(linear_combination_of_vectors,basis_vector)
        end
    end # this code block is used to find the linear combination of vectors of the system of linear equations, and store it in linear_combination_of_vectors

    return particular_vector, linear_combination_of_vectors   
        
end  
end

In [1]:
# Final attempt of QR decomposition

function Householder_matrix(v::Vector{<:Number}) 
    # it takes a vector and returns a corresponding householder matrix 
    sz = size(v)
    mag2 = 0
    I = zeros(Float64,sz[1],sz[1])
  @inbounds @simd  for i in 1:1:sz[1]  # construction of Identity matrix
        mag2 += v[i]^2
        I[i,i]=1
    end
    if v[1]>=0
        v[1] = v[1]+mag2^0.5
    else
        v[1] = v[1]-mag2^0.5  
    end   
    vTv = (transpose(v)*v)
    vvT = v*transpose(v)
    H = I - 2*(vvT/vTv) # construction of householder matrix

    return H
end
function QR(M_raw :: Matrix{<:Number})
    R = Float64.(M_raw)
    sz = size(R)
    pikachu = min(sz[1]-1,sz[2])  # max amount of rank possible
    #= diagonal for a non square matrix = minimum of rows and column, and thats exactly whats done here
        although we have modified it a bit for the tall matrices (having more rows then columns)
        now diagonal = minimum of (rows-1) and column, otherwise few columns would not be transformed accordingly
    =#
    Q = zeros(Float64,sz[1],sz[1])
 @inbounds @simd   for i in 1:1:sz[1]  # construction of Identity matrix { Q =I}
        Q[i,i]=1
    end
    
 @inbounds @simd   for i in 1:1:pikachu
        column_vec = @view R[i:sz[1],i] # selcting the column vector which is about to get boom-bam budup-budup booooooooooom
                                        # householder matrix will be constructed according to this vector
        #------------------------------------------------------------------------------------------------------
        hehe = Householder_matrix(copy(column_vec)) 
        R[i:sz[1],i:sz[2]] = hehe*R[i:sz[1],i:sz[2]] 
        #=householder matrix is applied to the effective matrix
        a householder matrix transforms the column vector (which is already selected above)
        hence the effective application of householder matrix is from the ith row to last row and ith column to last column
        {as the matrix before ith row and column is already converted into an upper triangular matrix)
        =#
       
        Q[:, i:sz[1]] = Q[:, i:sz[1]] * hehe # updating the Q matrix => Q = H1*H2*H3.......Hn
        
        #---------------------------------------------------------------------------------------------
        R[i+1:sz[1],i] .= 0.0 # without this also the algo will work, but it will have floating point errors
        # to make the elements below diagonal as zero, broadcasting{dot operator} "." is done
        # it is not necessary but it makes the matrix pleasing to eyes
    end
    return Q,R
end

QR (generic function with 1 method)

In [ ]:
# Attempted to give Q as a output with R ,but failed

function Householder_matrix(v::Vector{<:Number}) 
    # it takes a vector and returns a corresponding householder matrix 
    sz = size(v)
    mag2 = 0
    I = zeros(Float64,sz[1],sz[1])
    for i in 1:1:sz[1]  # construction of Identity matrix
        mag2 += v[i]^2
        I[i,i]=1
    end
    if v[1]>=0
        v[1] = v[1]+mag2^0.5
    else
        v[1] = v[1]-mag2^0.5  
    end   
    vTv = (transpose(v)*v)
    vvT = v*transpose(v)
    H = I - 2*(vvT/vTv) # construction of householder matrix

    return H
end
function QR(M_raw :: Matrix{<:Number})
    R = Float64.(M_raw)
    sz = size(R)
    pikachu = min(sz[1]-1,sz[2])  # max amount of rank possible
    #= diagonal for a non square matrix = minimum of rows and column, and thats exactly whats done here
        although we have modified it a bit for the tall matrices (having more rows then columns)
        now diagonal = minimum of (rows-1) and column, otherwise few columns would not be transformed accordingly
    =#
    I = zeros(Float64,sz[1],sz[1])
    for i in 1:1:sz[1]  # construction of Identity matrix
        I[i,i]=1
    end
    
    for i in 1:1:pikachu
        column_vec = @view R[i:sz[1],i] # selcting the column vector which is about to get boom-bam budup-budup booooooooooom
                                        # householder matrix will be constructed according to this vector
        #------------------------------------------------------------------------------------------------------
        hehe = Householder_matrix(copy(column_vec)) 
        R[i:sz[1],i:sz[2]] = hehe*R[i:sz[1],i:sz[2]] 
        #=householder matrix is applied to the effective matrix
        a householder matrix transforms the column vector (which is already selected above)
        hence the effective application of householder matrix is from the ith row to last row and ith column to last column
        {as the matrix before ith row and column is already converted into an upper triangular matrix)
        =#
        H_raw = zeros(Float64,sz[1],sz[1])
        for j in 1:1:i-1
            H_raw[j,j] = 1
        end
        H_raw[sz[1]-i+1:sz[1],sz[1]-i+1:sz[1]] = hehe
        I = I*H_raw
        
        #---------------------------------------------------------------------------------------------
        R[i+1:sz[1],i] .= 0.0 # without this also the algo will work, but it will have floating point errors
        # to make the elements below diagonal as zero, broadcasting{dot operator} "." is done
        # it is not necessary but it makes the matrix pleasing to eyes
    end
    return I,R
end

In [ ]:
# Second attempt to make a QR decomposition, now it outputs the R( upper triangular matrix)
function Householder_matrix(v::Vector{<:Number}) 
    # it takes a vector and returns a corresponding householder matrix 
    sz = size(v)
    mag2 = 0
    I = zeros(Float64,sz[1],sz[1])
    for i in 1:1:sz[1]  # construction of Identity matrix
        mag2 += v[i]^2
        I[i,i]=1
    end
    if v[1]>=0
        v[1] = v[1]+mag2^0.5
    else
        v[1] = v[1]-mag2^0.5  
    end   
    vTv = (transpose(v)*v)
    vvT = v*transpose(v)
    H = I - 2*(vvT/vTv) # construction of householder matrix

    return H
end
function QR(M_raw :: Matrix{<:Number})
    R = Float64.(M_raw)
    sz = size(R)
    pikachu = min(sz[1]-1,sz[2]) 
    #= diagonal for a non square matrix = minimum of rows and column, and thats exactly whats done here
        although we have modified it a bit for the tall matrices (having more rows then columns)
        now diagonal = minimum of (rows-1) and column, otherwise few columns would not be transformed accordingly
    =#
    
    for i in 1:1:pikachu
        column_vec = @view R[i:sz[1],i] # selcting the column vector which is about to get boom-bam budup-budup booooooooooom
                                        # householder matrix will be constructed according to this vector
        #------------------------------------------------------------------------------------------------------
        hehe = Householder_matrix(copy(column_vec)) 
        R[i:sz[1],i:sz[2]] = hehe*R[i:sz[1],i:sz[2]] 
        #=householder matrix is applied to the effective matrix
        a householder matrix transforms the column vector (which is already selected above)
        hence the effective application of householder matrix is from the ith row to last row and ith column to last column
        {as the matrix before ith row and column is already converted into an upper triangular matrix)
        =#
        #---------------------------------------------------------------------------------------------
       R[i+1:sz[1],i] .= 0.0 # without this also the algo will work, but it will have floating point errors
        # to make the elements below diagonal as zero, broadcasting{dot operator} "." is done
        # it is not necessary but it makes the matrix pleasing to eyes
    end
    return R
end

In [ ]:
# First attempt to make QR decomposition, total disaster

function QR(M_raw::Matrix{<:Number})
    M = Float64.(M_raw) 
    sz = size(M)
    j_carry =1
    i_carry =0
    while j_carry <=sz[2] #selects columns
        i = i_carry+1
        while i<=sz[1] #select the row
                col_vector = @view M[i:sz[1],j_carry] # selcting the column vector which is about to get boom-bam budup-budup booooooooooom
                mag = 0
                for comp in 1:1:sz[1] 
                    mag += col_vector[comp]^2
                end
                if col_vector[1] >0
                    col_vector[1] = col_vector[1]+mag^0.5
                else
                    col_vector[1] = col_vector[1]-mag^0.5
                end # by the end of this code we have the vector used for forming the respective householder matrix( v in H = I -2vvT/vTv)
                I = zeros(Float64,sz[1]+1-j_carry,sz[1]+1-j_carry)
                for modi in 1:1:sz[1]+1-j_carry
                    I[modi,modi] = 1
                end
                H_raw = I-(2/(transpose(col_vector)*col_vector))*col_vector*transpose(col_vector)
                H = zeros(Float64,sz[1],sz[1])
                for meow in 1:1:j_carry-1
                    H[meow,meow] =1
                end
                H[sz[1]-j_carry+1:sz[1],sz[1]-j_carry+1:sz[1]] = H_raw # Householder matrix formed here 
                H = I*H
                j_carry +=1
                M =H*M

            i_carry +=1
            j_carry+=1
                
        end
    end
    return M,H
end